# Frozen PCA/SVM held-out test
Owner: Felix / Zhangnan Fan. This notebook reads the completed test deliverable and its comparison with the two CNN pipelines. Settings were selected on validation: RGB 32×32, PCA 20 components, linear SVC C=0.1. Training used 19,124 images; the shared test manifest contains 4,110 images from 31 recorded patient ID groups.

The registered checkpoint in models/pca_linear_svm.joblib is the original saved training checkpoint, copied without refitting. All final numbers below are TEST results. Earlier validation artifacts remain separate.


In [ ]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'models/pca_linear_svm_metadata.json').is_file()
OUT = ROOT / 'artifacts/felix_svm/test'


## Optional execution of the fixed test entry point
Leave the switch off to read published results without image access. The command checks checkpoint/manifest/preprocessing hashes, replays validation predictions first, and does not fit PCA or SVM. Completed test results are reported without recomputing predictions.

For a checkout without exported results, set MALARIA_DATA_ROOT to the folder containing Parasitized/ and Uninfected/, then enable the switch. Do not use test results to choose new hyperparameters.


In [ ]:
RUN_FROZEN_TEST = False
if RUN_FROZEN_TEST:
    subprocess.run([sys.executable, str(ROOT/'scripts/evaluate_frozen_svm.py'),
                    '--confirm-frozen-protocol'], cwd=ROOT, check=True)
else:
    print('Reading exported test results; no model fitting or inference requested.')


## Metrics, shared comparison and error counts
Positive class: Parasitized=1. A false negative is an infected cell predicted as uninfected. The SVM decision score is not a calibrated probability. Compare complete pipelines: SVM inputs are 32px, CNN/ResNet inputs are 128px, and ResNet starts from ImageNet pretrained weights.


In [ ]:
metrics = json.loads((OUT/'test_metrics.json').read_text())
info = json.loads((OUT/'checkpoint_info.json').read_text())
assert info['model_refitted'] is False
assert info['checkpoint_file_unchanged']
assert metrics['n_images'] == 4110
comparison = pd.read_csv(OUT/'model_comparison.csv')
display(comparison[['model','split','accuracy','recall','f1','roc_auc','fn','fp']])
display(Image(filename=str(OUT/'confusion_matrix.png')))
print('SVM:', {k: metrics[k] for k in ['tn','fp','fn','tp']})


## Interpretation and limitations
Test accuracy is 65.67%, recall 60.90%, F1 0.6294 and ROC-AUC 0.7133. There are 769 false negatives and 642 false positives. Validation recall was 69.61%, so similar overall accuracy does not mean class-specific performance stayed the same.

The CNN pipelines have higher test F1 on this split. Different input resolution, preprocessing and pretrained initialization prevent attributing all differences to architecture alone. PCA maximizes image variance, which need not preserve infection-specific information; fine detail may be lost at 32px. These are limitations and hypotheses, not demonstrated causes of individual errors.
